### Make plotly plots for measuring features in solar wind data

In [14]:
import pickle
import os
import numpy as np
from sunpy.time import parse_time
import matplotlib.pyplot as plt

data_path='data/'

#Plotly imports
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.io as pio
from plotly.offline import iplot, init_notebook_mode
import plotly.express as px
import pandas as pd
from datetime import datetime

pio.renderers.default = 'browser'

#### print files in data_path

In [2]:
files = os.listdir(data_path)
files.sort()
fnames=[os.path.join(data_path, f) for f in files]
for item in fnames:
    print(item)

data/.DS_Store
data/.ipynb_checkpoints
data/CME_events_times_new.csv
data/GS_events_times.csv
data/GS_events_times_misses.csv
data/earth_pos_HEEQ_20221111_20240604.p
data/indices_geomagnetic_storms_beacon_plasma.p
data/l1_HEEQ_20221111_20240604.p
data/l1_symh_fin_new.p
data/mean_time_shift_and_sigma_beacon_plasma_new.p
data/noaa_20221111_20240604_B_GSM_Coords_GSE.p
data/noaa_archive_gsm.p
data/omni_20221111_20240604.p
data/sta_beacon_20221101_20240606_gsm.p
data/sta_beacon_20221111_20240604_B_GSM_Coords_GSE_new.p
data/sta_beacon_plasma_science_20221111_20240605_GSM.p
data/sta_science_20221111_20240604_B_GSM_Coords_GSE_new.p
data/sta_science_HEEQ_20221111_20240604.p
data/sta_symh_final_beacon_plasma.p
data/stereoa_2007_now_rtn.p


In [19]:
url='https://helioforecast.space/static/sync/icmecat/HELIO4CAST_ICMECAT_v23.csv'
ic=pd.read_csv(url)
ic.keys()

ic.icme_start_time = pd.to_datetime(ic.icme_start_time, format='%Y-%m-%dT%H:%MZ')
ic.mo_start_time = pd.to_datetime(ic.mo_start_time, format='%Y-%m-%dT%H:%MZ')
ic.mo_end_time = pd.to_datetime(ic.mo_end_time, format='%Y-%m-%dT%H:%MZ')


In [21]:
ic_wind_all = ic[ic.sc_insitu=='Wind']
ic_sta_all = ic[ic.sc_insitu=='STEREO-A']

ic_wind = ic_wind_all[(ic_wind_all.icme_start_time >= datetime(2022,11,1)) & (ic_wind_all.mo_end_time <= datetime(2024,7,1)) ]
ic_sta = ic_sta_all[(ic_sta_all.icme_start_time >= datetime(2022,11,1)) & (ic_sta_all.mo_end_time <= datetime(2024,7,1)) ]

In [31]:
times = pd.read_csv(data_path+'CME_events_times_new.csv', header=0, delimiter=';')
times.sta_start = pd.to_datetime(times.sta_start, format='%Y-%m-%dT%H:%MZ')
times.sta_end = pd.to_datetime(times.sta_end, format='%Y-%m-%dT%H:%MZ')
times.l1_start = pd.to_datetime(times.l1_start, format='%Y-%m-%dT%H:%MZ')
times.l1_end = pd.to_datetime(times.l1_end, format='%Y-%m-%dT%H:%MZ')

In [22]:
print('STEREO-A mag & plasma data')
#file2='noaa_20221111_20240604_B_GSM_Coords_GSE_corr.p'
file='sta_beacon_plasma_science_20221111_20240605_GSM.p'
sc=pickle.load(open(data_path+file, "rb" ) ) 

print('NOAA mag & plasma data')
file2='noaa_20221111_20240604_B_GSM_Coords_GSE.p'
sc2=pickle.load(open(data_path+file2, "rb" ) ) 

#print('shifted STEREO-A data (to Earth)')
#sc3 = pickle.load(open(data_path+'sta_symh_final.p', 'rb'))
#[sc, header]=pickle.load(open(data_path+file, "rb" ) )  


print('done')

STEREO-A mag & plasma data
NOAA mag & plasma data
done


#### Use plotly for measuring times

#### with plasma data


In [29]:
fig = go.Figure()

fig.add_trace(go.Scatter(x=sc.time, y=sc.bx, name='Bx',line_color='red'))
fig.add_trace(go.Scatter(x=sc.time, y=sc.by, name='By',line_color='green'))
fig.add_trace(go.Scatter(x=sc.time, y=sc.bz, name='Bz',line_color='blue'))
fig.add_trace(go.Scatter(x=sc.time, y=sc.bt, name='Bt',line_color='black'))
for i in range(len(ic_sta.icme_start_time)):
    fig.add_trace(go.Scatter(x=[ic_sta.icme_start_time.iloc[i],ic_sta.icme_start_time.iloc[i]], y=[-100,100], name='Bt',line_color='grey'))
    fig.add_trace(go.Scatter(x=[ic_sta.mo_end_time.iloc[i],ic_sta.mo_end_time.iloc[i]], y=[-100,100], name='Bt',line_color='black'))
fig.show()

In [33]:
nrows=4
#init_notebook_mode(connected = True)
#init_notebook_mode(connected = False)

#fig=plt.figure(figsize=(10,6), dpi=150)

fig = make_subplots(rows=nrows, cols=1, shared_xaxes=True)


#for column, color in zip(['b_x', 'b_y', 'b_z', 'b_tot'], ['red', 'green', 'blue', 'black']):
fig.add_trace(go.Scatter(x=sc.time, y=sc.bx, name='Bx',line_color='red'), row=1, col=1)
fig.add_trace(go.Scatter(x=sc.time, y=sc.by, name='By',line_color='green'), row=1, col=1)
fig.add_trace(go.Scatter(x=sc.time, y=sc.bz, name='Bz',line_color='blue'), row=1, col=1)
fig.add_trace(go.Scatter(x=sc.time, y=sc.bt, name='Bt',line_color='black'), row=1, col=1)
for i in range(len(ic_sta.icme_start_time)):
    fig.add_trace(go.Scatter(x=[ic_sta.icme_start_time.iloc[i],ic_sta.icme_start_time.iloc[i]], y=[-100,100], name='Bt',line_color='black'), row=1, col=1)
    fig.add_trace(go.Scatter(x=[ic_sta.mo_end_time.iloc[i],ic_sta.mo_end_time.iloc[i]], y=[-100,100], name='Bt',line_color='black'), row=1, col=1)
for i in range(len(times.sta_start)):
    fig.add_trace(go.Scatter(x=[times.sta_start.iloc[i],times.sta_start.iloc[i]], y=[-100,100], name='Bt',line_color='red'), row=1, col=1)
    fig.add_trace(go.Scatter(x=[times.sta_end.iloc[i],times.sta_end.iloc[i]], y=[-100,100], name='Bt',line_color='red'), row=1, col=1)

#fig.add_trace(go.Scatter(x=sc3['time'], y=sc3['bx'], name='Bx - shifted',line_color='red'), row=2, col=1)
#fig.add_trace(go.Scatter(x=sc3['time'], y=sc3['by'], name='By - shifted',line_color='green'), row=2, col=1)
#fig.add_trace(go.Scatter(x=sc3['time'], y=sc3['bz'], name='Bz - shifted',line_color='blue'), row=2, col=1)
#fig.add_trace(go.Scatter(x=sc3['time'], y=sc3['bt'], name='Bt - shifted',line_color='black'), row=2, col=1)

fig.add_trace(go.Scatter(x=sc2.time, y=sc2.bx, name='Bx - NOAA',line_color='red'), row=2, col=1)
fig.add_trace(go.Scatter(x=sc2.time, y=sc2.by, name='By - NOAA',line_color='green'), row=2, col=1)
fig.add_trace(go.Scatter(x=sc2.time, y=sc2.bz, name='Bz - NOAA',line_color='blue'), row=2, col=1)
fig.add_trace(go.Scatter(x=sc2.time, y=sc2.bt, name='Bt - NOAA',line_color='black'), row=2, col=1)
for i in range(len(ic_wind.icme_start_time)):
    fig.add_trace(go.Scatter(x=[ic_wind.icme_start_time.iloc[i],ic_wind.icme_start_time.iloc[i]], y=[-100,100], name='Bt',line_color='grey'), row=2, col=1)
    fig.add_trace(go.Scatter(x=[ic_wind.mo_end_time.iloc[i],ic_wind.mo_end_time.iloc[i]], y=[-100,100], name='Bt',line_color='black'), row=2, col=1)
for i in range(len(times.l1_start)):
    fig.add_trace(go.Scatter(x=[times.l1_start.iloc[i],times.l1_start.iloc[i]], y=[-100,100], name='Bt',line_color='red'), row=2, col=1)
    fig.add_trace(go.Scatter(x=[times.l1_end.iloc[i],times.l1_end.iloc[i]], y=[-100,100], name='Bt',line_color='red'), row=2, col=1)
    
#fig.add_trace(go.Scatter(x=sc.time, y=sc.vt, name='Vt',line_color='black'), row=2, col=1)
fig.add_trace(go.Scatter(x=sc.time, y=sc.vt, name='Vt',line_color='black'), row=3, col=1)
fig.add_trace(go.Scatter(x=sc2.time, y=sc2.vt, name='Vt - NOAA',line_color='red'), row=3, col=1)
fig.add_trace(go.Scatter(x=sc.time, y=sc.np, name='Np',line_color='black'), row=4, col=1)
fig.add_trace(go.Scatter(x=sc2.time, y=sc2.np, name='Np - NOAA',line_color='red'), row=4, col=1)
#fig.add_trace(go.Scatter(x=sc2.time, y=sc2.tp, name='Tp',line_color='black'), row=4, col=1)

fig.update_layout(title='sub-L1, STEREO-A')

fig.write_html('figures/l1_sta_cme_events.html')
fig.show()

### Plot observed SYM-H indices 

In [8]:
omni_input = pickle.load(open(data_path+'omni_20221111_20240604.p', 'rb'))

In [9]:
nrows=1
#init_notebook_mode(connected = True)
#init_notebook_mode(connected = False)

#fig=plt.figure(figsize=(10,6), dpi=150)

fig = make_subplots(rows=nrows, cols=1, shared_xaxes=True)


#for column, color in zip(['b_x', 'b_y', 'b_z', 'b_tot'], ['red', 'green', 'blue', 'black']):
fig.add_trace(go.Scatter(x=omni_input.time, y=omni_input.symh, name='SYM-H',line_color='black'), row=1, col=1)